# FMDV model

The model is based on the model in [Jolles et al, 2021](https://science.org/doi/10.1126/science.abd2475), simplified by (in order of decreasing complexity):

1. All waiting times are age independent. In the original, death and birth were age dependent.
2. All waiting times are exponential distributions. In the original, waning of maternal immunity, progression, and recovery from acute infection were gamma distributions.
3. The birth rate is constant in time. In the original, the birth rate is seasonal.
4. Newly infected hosts are immediately infectious. In the orignal, newly infected hosts have a short delay before becoming infectious, in the exposed class.

diagram_standalone.svg

## Parameter values

These are mostly from the supplmental material to Jolles et al, 2021.

| Parameter                     |          all |          SAT1 |          SAT2 |          SAT3 |
|:------------------------------|--------------:|--------------:|--------------:|--------------:|
| Death mean waiting time       |   4.387 year  |             — |             — |             — |
| Birth mean waiting time       |   4.387 year  |             — |             — |             — |
| Waning mean waiting time      |   0.37  year  |             — |             — |             — |
| Acute transmission rate       |             — |   2.8   day⁻¹ |   1.6   day⁻¹ |   1.2   day⁻¹ |
| Carrier transmission rate     |             — |   0.028 day⁻¹ |   0.003 day⁻¹ |   0.012 day⁻¹ |
| Progression mean waiting time |             — |   0.5   day   |   1.3   day   |   2.8  day    |
| Recovery mean waiting time    |             — |   5.7  day    |   4.6  day    |   4.2  day    |
| Carrier probability           |             — |   0.90        |   0.44        |   0.67        |
| Carrier recovery              |             — | 243    day    | 180    day    | 174    day    |

**Note the different time units!**

We will take all of the waiting times to be exponential distributions. In Jolles et al, 2021, all except death & birth were gamma distributions, which have an extra shape parameter.

### Death

Death in Jolles et al, 2021 was age dependent, from the estimated age-dependent annual survival. Here, the mean waiting time is the mean lifetime calculated from that age-dependent annual survival.

### Birth

Birth in Jolles et al, 2021 was age dependent (menarchy at age 4 years) and seasonal, but scaled so that the expected population size is constant in time. Here, the mean waiting time is the same as for death so that the expected population size is constant in time. (In Jolles et al, 2021, the sex of hosts was also tracked and only females gave birth. Here, we will not track sex.)

### Infection

The force of infection is
$$
\lambda_\text{infection} =
\frac{\beta_\text{acute}}{N} I
+ \frac{\beta_\text{carrier}}{N} C,
$$
where $\beta$ are the transmission rates, $I$ is the number of infectious hosts, $C$ is the number of carrier hosts, and $N$ is the **initial** population size. The mean waiting time for infection is then
$$
\frac{1}{\lambda_\text{infection}}.
$$
We chose to scale the transmission rates by initial population size because we first chose the density-dependent force of infection and then wanted $R_0$ to be independent of initial population size. (The logic is a bit muddled. An alternative would be use the frequency-dependent force of infection using the same formula but with $N$ being the **current** population size.)

### Recovery

Acute infection in Jolles et al, 2021 was split between a first exposed class followed by the infectious class. The transition from exposed to infectious was called progression and the transition out of infectious was called recovery. Here, we are assuming there is no exposed class. **For our mean time to recovery, should we use their mean time to recovery or the sum of their mean times to progression and recovery?** My first instinct is to use just their mean time to recovery as it keeps $R_0$ (approximately) constant between the two models. If we use the sum, we should consider scaling the transmission rates to keep $R_0$ (approximately) constant.

# Gillespie's next-reaction method

In [3]:
import abc
import collections

import numpy
import pandas

[Gillespie, 1977](https://doi.org/10.1021/j100540a008),
[Gibson & Bruck, 2000](https://doi.org/10.1021/jp993732q),
[Gillespie, 2007](https://doi.org/10.1146/annurev.physchem.58.032806.104637)

The function `simulate()` implements the algorithm in Python.

In [4]:
def simulate(events, time_span, state_start, seed=None):
    '''Simulate with Gillespie's next-reaction method.
    `events` is a list of all the events that can occur.
    `time_span` is `(time_start, time_end)`.
    `state_start` is the state at `time_start`.
    `seed` is for the random-number generator.'''
    # Simplify by ensuring events is not empty
    assert len(events) > 0
    (time, time_end) = time_span
    # Simplify by ensuring `time_end` is finite
    assert numpy.isfinite(time_end)
    state = state_start.copy()
    # Build the random-number generator
    rng = numpy.random.default_rng(seed)
    # Initialize storage of the simulation history
    history = History()
    while True:
        history.update(time, state)
        if time >= time_end:
            break  # Exit the `while` loop
        waiting_times = [
            event.get_waiting_time(state, rng)
            for event in events
        ]
        time_next = time + min(waiting_times)
        if time_next <= time_end:
            time = time_next
            # Find the position of the minimum in `waiting_times`
            # to find the corresponding event in `events`
            i_next = numpy.argmin(waiting_times)
            event_next = events[i_next]
            state_change = event_next.get_state_change(state, rng)
            state.update(state_change)
        else:
            # The next event is after `time_end`
            time = time_end
    return history.to_dataframe()

The exponential distribution is *memoryless*: the distribution given that the event has not yet occurred is the same as the orignal distribution. E.g., under the assumption that time to death is exponential, living buffalo with different birth dates all have the same distribution of time from now until death.

Assuming that the waiting times for all events are exponential, combined with the events being age independent, means we only need to know a buffalo's current compartment ('maternal_immunity' or 'susceptible' or ...) to accurately determine their future. Thus we can use the current count in each compartment as the `state` of the population. `collections.Counter()` is a convenient way to store those counts.

In [21]:
state = collections.Counter({
    'maternal_immunity': 10,
    'susceptible': 50,
    'infectious': 2,
    'carrier': 20,
    'recovered': 100,
})
state

Counter({'recovered': 100,
         'susceptible': 50,
         'carrier': 20,
         'maternal_immunity': 10,
         'infectious': 2})

In [6]:
# The state change for a waning event
state_change = collections.Counter({
    'maternal_immunity': -1,
    'susceptible': +1,
})

# Apply the change
state.update(state_change)
state

Counter({'recovered': 100,
         'susceptible': 51,
         'carrier': 20,
         'maternal_immunity': 9,
         'infectious': 2})

The elements of `events` must look like `Event()` below. (Ignore the `abc` things for now.)

In [7]:
class Event(metaclass=abc.ABCMeta):
    '''Base for a kind of event.'''

    @abc.abstractmethod
    def get_waiting_time(self, state, rng):
        '''Get a random sample of the waiting time to the next event.'''
        raise NotImplementedError

    @abc.abstractmethod
    def get_state_change(self, state, rng):
        '''Get the change to `state` from this event occurring.'''
        raise NotImplementedError

For example, here is the waning event.

(Another simplification from the exponential distribution is that, if the waiting time for a single buffalo to undergo an event is exponential with mean $\mu$, the waiting time for the first of $n$ buffalo to undergo the event is exponential with mean $\mu / n$.)

In [22]:
class Waning(Event):
    '''Waning of 'maternal_immunity' to 'susceptible'.'''

    state_change = collections.Counter({
        'maternal_immunity': -1,
        'susceptible': +1,
    })

    def __init__(self, parameters):
        self.mean_waiting_time = parameters['waning_mean_waiting_time']

    def get_waiting_time(self, state, rng):
        count = state['maternal_immunity']
        try:
            mean = self.mean_waiting_time / count
        except ZeroDivisionError:
            return numpy.inf
        return rng.exponential(mean)

    def get_state_change(self, state, rng):
        return self.state_change

Replicate the waning event for birth, death, infection, and recovery

In [16]:
class Birth(Event):
    '''Birth of new individuals into either 'maternal_immunity' or 'susceptible'.'''

    def __init__(self, parameters):
        self.mean_waiting_time = parameters['birth_mean_waiting_time']

    def get_waiting_time(self, state, rng):
        count = state.total()
        
        try:
            mean = self.mean_waiting_time / count
        except ZeroDivisionError:
            return numpy.inf
        return rng.exponential(mean)

    def get_state_change(self, state, rng):
        n = state.total()
        p_maternal = (state['carrier'] + state['recovered']) / n
        birth = rng.choice(['maternal_immunity', 'susceptible'], 1, p=[p_maternal, 1-p_maternal])
        self.state_change = collections.Counter({
            birth[0]: +1,
        })
        return self.state_change

In [17]:
class Death(Event):
    '''Death of an individual from any class.'''

    def __init__(self, parameters):
        self.mean_waiting_time = parameters['death_mean_waiting_time']

    def get_waiting_time(self, state, rng):
        count = state.total()
        
        try:
            mean = self.mean_waiting_time / count
        except ZeroDivisionError:
            return numpy.inf
        return rng.exponential(mean)

    def get_state_change(self, state, rng):
        n = state.total()
        death = rng.choice(['maternal_immunity', 'susceptible', 'infectious', 'carrier', 'recovered'],
                           1, 
                           p=[state['maternal_immunity']/n, state['susceptible']/n, state['infectious']/n, state['carrier']/n, state['recovered']/n])
        self.state_change = collections.Counter({
            death[0]: -1,
        })
        return self.state_change

In [37]:
class Infection_SAT1(Event):
    '''Infection of an individual from susceptible class.'''

    state_change = collections.Counter({
        'susceptible': -1,
        'infectious': +1,
    })

    def __init__(self, parameters, state):
        self.beta_acute = parameters['sat1_acute_transmission_rate'] / state.total()
        self.beta_carrier = parameters['sat1_carrier_transmission_rate'] / state.total()

    def get_waiting_time(self, state, rng):
        
        count = state['susceptible']
        
        try:
            foi = self.beta_acute * state['infectious'] + self.beta_carrier * state['carrier']
            self.mean_waiting_time = 1 / foi
            mean = self.mean_waiting_time / count
        except ZeroDivisionError:
            return numpy.inf
        return rng.exponential(mean)

    def get_state_change(self, state, rng):
        return self.state_change

In [32]:
class Recovery_SAT1(Event):
    '''Infection of an individual from susceptible class.'''

    def __init__(self, parameters):
        self.mean_waiting_time = parameters['sat1_recovery_mean_waiting_time']
        self.carrier_probability = parameters['sat1_carrier_probability']

    def get_waiting_time(self, state, rng):
        count = state['infectious']
        
        try:
            mean = self.mean_waiting_time / count
        except ZeroDivisionError:
            return numpy.inf
        return rng.exponential(mean)

    def get_state_change(self, state, rng):
        recovery_class = rng.choice(['carrier', 'recovered'],
                           1, 
                           p=[self.carrier_probability, 1-self.carrier_probability])
        self.state_change = collections.Counter({
            'infectious': -1,
            recovery_class[0]: +1,
        })
        return self.state_change

In [34]:
class Carrier_Recovery_SAT1(Event):
    '''Infection of an individual from susceptible class.'''
    state_change = collections.Counter({
        'carrier': -1,
        'recovered': +1,
    })

    def __init__(self, parameters):
        self.mean_waiting_time = parameters['sat1_carrier_recovery_mean_waiting_time']

    def get_waiting_time(self, state, rng):
        count = state['carrier']
        
        try:
            mean = self.mean_waiting_time / count
        except ZeroDivisionError:
            return numpy.inf
        return rng.exponential(mean)

    def get_state_change(self, state, rng):
       return self.state_change

Of less importance, here's a version of the object that stores the simulation history.

In [35]:
class History:
    '''Record simulation history.'''

    def __init__(self):
        self.time = []
        self.state = []

    def update(self, time, state):
        '''Update for the current `time` and `state`.'''
        self.time.append(time)
        # Must `.copy()` to record current, not future, values
        self.state.append(state.copy())

    def to_dataframe(self):
        '''Convert to something more useful.'''
        index = pandas.Index(self.time, name='time')
        return pandas.DataFrame(self.state, index=index)

Here's a quick example of running `simulate()`.

In [41]:
parameters = {
    'birth_mean_waiting_time': 4.387*365.25,    # units: day
    'death_mean_waiting_time': 4.387*365.25,    # units: day
    'waning_mean_waiting_time': 0.37*365.25,    # units: day
    
    'sat1_acute_transmission_rate': 2.8,        # units: 1/day
    'sat1_carrier_transmission_rate': 0.028,    # units: 1/day
    'sat1_progression_mean_waiting_time': 0.5,  # units: day
    'sat1_recovery_mean_waiting_time': 5.7,     # units: day
    'sat1_carrier_probability': 0.9,            # units: none
    'sat1_carrier_recovery_mean_waiting_time': 243,  # units: day
    
    'sat2_acute_transmission_rate': 1.6,        # units: 1/day
    'sat2_carrier_transmission_rate': 0.003,    # units: 1/day
    'sat2_progression_mean_waiting_time': 1.3,  # units: day
    'sat2_recovery_mean_waiting_time': 4.6,     # units: day
    'sat2_carrier_probability': 0.44,           # units: none
    'sat2_carrier_recovery_mean_waiting_time': 180,  # units: day
    
    'sat3_acute_transmission_rate': 1.2,        # units: 1/day
    'sat3_carier_transmission_rate': 0.012,     # units: 1/day
    'sat3_progression_mean_waiting_time': 2.8,  # units: day
    'sat3_recovery_mean_waiting_time': 4.2,     # units: day
    'sat3_carrier_probability': 0.67,           # units: none
    'sat3_carrier_recovery_mean_waiting_time': 174,  # units: day
}

# `events` is a list of (instances) of kinds of `Event()`
events = (
    Waning(parameters),
    Birth(parameters),
    Death(parameters),
    Infection_SAT1(parameters, state),
    Recovery_SAT1(parameters),
    Carrier_Recovery_SAT1(parameters),
    # etc
)

result = simulate(events, (0, 10000), state)
result

,maternal_immunity,susceptible,infectious,carrier,recovered
time,,,,,
0.000000,10,50,2,20,100
0.087439,10,49,3,20,100
0.236513,10,49,2,20,101
0.974984,10,49,1,21,101
1.102175,10,48,2,21,101
...,...,...,...,...,...
9987.177751,20,7,0,13,148
9993.431644,20,7,0,13,147
9997.208097,21,7,0,13,147


The only `Event()` we have included so far is `Waning()`, so the only changes in the result are the number in 'maternal_immunity' decreasing by one and the number in 'susceptible' increasing by one at each event (plus the last entry at `time_end = 10`).

## Later improvements

1. (minor) Handle the (very unlikely) possibility that multiple events have waiting time `min(waiting_times)` or at least raise an error if it ever happens. Here's what I've used before:
```
def min_and_argmin(self, arr):
    '''Find the minimum of `arr` and its position.
    If the minimum occurs in more than one position,
    randomly choose one.'''
    val = arr.min(initial=math.inf)
    if val < math.inf:
        mask = arr == val
    else:
        mask = numpy.isposinf(arr)
    (idxs,) = mask.nonzero()
    if len(idxs) == 0:
        idx = None
    elif len(idxs) == 1:
        (idx,) = idxs
    else:
        idx = self.rng.choice(idx)
    return (val, idx)
```

2. Consider adding a stopping condition beyond `time >= time_end`, like if the pathogen goes extinct.
3. Consider reducing the amount of information recorded by `History()`.
